# AI Stylist: train the clothing classifier (Tier 1)

Trains a small on-device model that suggests a garment's **type** (and so its category) and
**pattern**, from one photo. The app shows these as editable suggestions; manual entry always
works. Decisions: D-08 and P-3 in `DECISIONS.md`.

**Before you start:** *Runtime → Change runtime type → GPU* (T4 is enough).

**Time:** downloading and preparing the data takes roughly 30–60 minutes; training roughly
1–3 hours on a T4, depending on settings. Outputs are saved to Google Drive, so a disconnect
loses at most the current step.

**Licences:** Fashionpedia annotations are CC BY 4.0 (attribution required in the app). The
photos are not owned by Fashionpedia; they come from Flickr, Unsplash, Pexels and similar
sites under those sites' terms. We use them only to train; no photo is shipped. The prepare
step reports the image licences it saw.

In [ ]:
# 1. Check the GPU and library versions.
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > GPU"
import tensorflow as tf, keras, sys
print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2. Keep outputs on Google Drive (survives disconnects).
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ai_stylist_training"
!mkdir -p {DRIVE}/annotations {DRIVE}/runs

In [ ]:
# 3. Get the training code (public repo; only the training/ folder is used).
BRANCH = "feature/m4-classifier-training"  # change to "main" once merged
!rm -rf /content/ai-stylist
!git clone --depth 1 --branch {BRANCH} https://github.com/amarhbaneen/ai-stylist.git /content/ai-stylist
%cd /content/ai-stylist/training
!python -m unittest discover -s tests -v

## Data

Official links from the Fashionpedia README (github.com/cvdfoundation/fashionpedia).
Annotations are cached on Drive; images go to the Colab disk (faster, and they are not kept).
The official **val** set is used as our **test** set; 10% of train photos become **val**.

In [ ]:
# 4. Download annotations (cached on Drive) and images (Colab disk).
import os
BASE = "https://s3.amazonaws.com/ifashionist-dataset"
for name in ["instances_attributes_train2020.json", "instances_attributes_val2020.json"]:
    if not os.path.exists(f"{DRIVE}/annotations/{name}"):
        !wget -q --show-progress -O {DRIVE}/annotations/{name} {BASE}/annotations/{name}
!mkdir -p /content/data
!wget -q --show-progress -c -O /content/data/train2020.zip {BASE}/images/train2020.zip
!wget -q --show-progress -c -O /content/data/val_test2020.zip {BASE}/images/val_test2020.zip
!df -h /content | tail -1

In [ ]:
# 5. Unzip (prints the folder names so the next step uses the right paths).
!cd /content/data && unzip -q -o train2020.zip && unzip -q -o val_test2020.zip
!ls /content/data
!ls /content/data/* | head -3

In [ ]:
# 6. Cut one-garment crops. Fails with a clear message if a label name in
#    config/label_map.json is not in the annotation file.
TRAIN_IMAGES = "/content/data/train"  # adjust if step 5 shows another folder name
TEST_IMAGES = "/content/data/test"    # the val/test zip; adjust if needed
CROPS = "/content/crops"
!python -m ai_stylist_training.prepare_data --annotations {DRIVE}/annotations/instances_attributes_train2020.json --images {TRAIN_IMAGES} --out {CROPS} --split train --holdout 0.1
!python -m ai_stylist_training.prepare_data --annotations {DRIVE}/annotations/instances_attributes_val2020.json --images {TEST_IMAGES} --out {CROPS} --split test

In [ ]:
# 7. Review: how pattern attributes were mapped, class counts, image licences.
import json
report = json.load(open(f"{CROPS}/train_report.json"))
print(json.dumps(report["resolved"]["patternAttributes"], indent=1))
print(json.dumps(report["crops"], indent=1))
print(json.dumps(report["patterns"], indent=1))
print(json.dumps(report["imageLicences"], indent=1))
print("skipped:", report["skipped"])

## Train, evaluate, export

Check the mapping above first. If a pattern attribute landed in the wrong class, edit
`config/label_map.json` (`patternRules`), then re-run step 6.

In [ ]:
# 8. Train. Defaults: MobileNetV3-Large, 4 epochs heads-only + up to 12 fine-tuning.
RUN = f"{DRIVE}/runs/run1"
!python -m ai_stylist_training.train --crops {CROPS} --out {RUN}

In [ ]:
# 9. Evaluate on the test set and calibrate confidences.
!python -m ai_stylist_training.evaluate --crops {CROPS} --run {RUN}

In [ ]:
# 10. Export to int8 LiteRT and check it against the Keras model.
!python -m ai_stylist_training.export_tflite --crops {CROPS} --run {RUN} --model-id garment-tier1 --version 1.0.0
!ls -la {RUN}

In [ ]:
# 11. Download what the app needs (also kept in Drive under runs/run1).
from google.colab import files
for name in ["garment-tier1-int8.tflite", "labels.json", "model_card.json", "eval_report.json"]:
    files.download(f"{RUN}/{name}")

## Next

Send back `garment-tier1-int8.tflite`, `labels.json`, `model_card.json` and `eval_report.json`.
They get checked (accuracy, size, SHA-256) before the model is added to the app.